# Lev week 3 experiments

Цель недели: рядом с CatBoost-экспериментом проверить ещё несколько моделей на тех же комбинированных признаках.

Проверяем не RandomForest, а отдельные альтернативы:

- `DecisionTreeRegressor` с `splitter="random"` как одно случайное дерево;
- `LinearRegression` со scaling.

`GradientBoostingRegressor` здесь не запускаем: на этих признаках sklearn-овый бустинг учится слишком долго, а отдельный CatBoost-эксперимент уже был в `lev_week3_ testCatboost.ipynb`.

Важно: это ровно стадия `lev_week3_ testCatboost.ipynb`. Берём агрегированные признаки Week 2, окно `50`, добавляем лаговые окна `10`, `20`, `30` и исключаем сенсоры как у Дениса. Shift-калибровку и Week 4 robustness сюда не добавляем, потому что тогда сравнение уже будет не с CatBoost-стадией.


## 1. Imports and configuration


In [19]:
import itertools
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

RANDOM_STATE = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Полный перебор специально большой, примерно как в CatBoost-ноутбуке.
# Для быстрой проверки можно поставить MAX_MODELS_TO_RUN = 3 или 5.
MAX_MODELS_TO_RUN = None


## 2. Загрузка данных

Используем FD001 из NASA C-MAPSS. Для train считаем RUL как разницу между последним циклом двигателя и текущим циклом. Для test берём официальный RUL из `RUL_FD001.txt`, как и в предыдущих ноутбуках.


In [20]:
columns = [
    "unit_id", "cycle", "op_setting_1", "op_setting_2", "op_setting_3",
    "sensor_1", "sensor_2", "sensor_3", "sensor_4", "sensor_5",
    "sensor_6", "sensor_7", "sensor_8", "sensor_9", "sensor_10",
    "sensor_11", "sensor_12", "sensor_13", "sensor_14", "sensor_15",
    "sensor_16", "sensor_17", "sensor_18", "sensor_19", "sensor_20", "sensor_21",
]

train_df = pd.read_csv("CMAPSSData/train_FD001.txt", sep=r"\s+", header=None, names=columns)
test_df = pd.read_csv("CMAPSSData/test_FD001.txt", sep=r"\s+", header=None, names=columns)
rul_test_df = pd.read_csv("CMAPSSData/RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL"])

target_col = "Remaining Useful Life"
train_df[target_col] = train_df["unit_id"].map(train_df.groupby("unit_id")["cycle"].max()) - train_df["cycle"]

feature_cols = [
    col for col in train_df.columns
    if col.startswith("op_setting_") or col.startswith("sensor_")
]

print(f"train rows: {len(train_df):,}")
print(f"test rows: {len(test_df):,}")
print(f"official test units: {len(rul_test_df):,}")
print(f"base feature columns: {len(feature_cols)}")


train rows: 20,631
test rows: 13,096
official test units: 100
base feature columns: 24


## 3. Агрегированные признаки Week 2

Здесь повторяем блок признаков из CatBoost-эксперимента. База та же: обязательные статистики и лучшие optional-пары, которые уже были выбраны на второй неделе.


In [21]:
extended_agg_window_size = 50
edge_window_size = 10
extended_agg_feature_cols = feature_cols
eps = 1e-8

mandatory_feature_blocks = [
    "mean",
    "std",
    "min",
    "max",
    "range",
    "delta",
    "slope",
    "last_minus_mean",
]

optional_feature_pairs = {
    "median_iqr": ["median", "iqr"],
    "last_first": ["last", "first"],
    "relative_changes": ["relative_delta", "relative_last_minus_mean"],
    "edge_means": ["mean_last_10", "mean_first_10"],
    "edge_change_volatility": ["mean_last_10_minus_first_10", "std_last_10"],
    "tail_quantiles": ["q10", "q90"],
    "robust_spread": ["q90_q10_range", "mad"],
    "energy_acceleration": ["rms", "slope_change"],
}

best_week2_optional_pairs = [
    "edge_means",
    "energy_acceleration",
    "last_first",
    "median_iqr",
    "relative_changes",
]

best_week2_optional_blocks = list(itertools.chain.from_iterable(
    optional_feature_pairs[pair_name]
    for pair_name in best_week2_optional_pairs
))
best_week2_blocks = mandatory_feature_blocks + best_week2_optional_blocks

print(f"selected aggregated blocks: {len(best_week2_blocks)}")


selected aggregated blocks: 18


In [22]:
def calculate_slope(values):
    x = np.arange(len(values))
    x = x - x.mean()
    denominator = (x ** 2).sum()
    if denominator == 0:
        return 0
    return np.dot(values, x) / denominator


def calculate_first_edge_mean(values):
    return values[:edge_window_size].mean()


def calculate_mad(values):
    median_value = np.median(values)
    return np.median(np.abs(values - median_value))


def calculate_rms(values):
    return np.sqrt(np.mean(values ** 2))


def calculate_slope_change(values):
    split_index = len(values) // 2
    if split_index == 0 or split_index == len(values):
        return 0
    return calculate_slope(values[split_index:]) - calculate_slope(values[:split_index])


def rename_feature_block(block, suffix):
    renamed_block = block.copy()
    renamed_block.columns = [
        f"{col}_{suffix}_{extended_agg_window_size}"
        for col in renamed_block.columns
    ]
    return renamed_block


In [23]:
def build_all_agg_window_features(source_df, include_target=False, require_full_window=False):
    sorted_df = source_df.sort_values(["unit_id", "cycle"]).reset_index(drop=True)
    min_periods = extended_agg_window_size if require_full_window else 1
    edge_min_periods = edge_window_size if require_full_window else 1

    rolling_features = sorted_df.groupby("unit_id")[extended_agg_feature_cols].rolling(
        window=extended_agg_window_size,
        min_periods=min_periods,
    )
    edge_rolling_features = sorted_df.groupby("unit_id")[extended_agg_feature_cols].rolling(
        window=edge_window_size,
        min_periods=edge_min_periods,
    )

    mean_features = rolling_features.mean().reset_index(level=0, drop=True)
    std_features = rolling_features.std(ddof=0).reset_index(level=0, drop=True).fillna(0)
    min_features = rolling_features.min().reset_index(level=0, drop=True)
    max_features = rolling_features.max().reset_index(level=0, drop=True)
    median_features = rolling_features.median().reset_index(level=0, drop=True)
    q10_features = rolling_features.quantile(0.10).reset_index(level=0, drop=True)
    q25_features = rolling_features.quantile(0.25).reset_index(level=0, drop=True)
    q75_features = rolling_features.quantile(0.75).reset_index(level=0, drop=True)
    q90_features = rolling_features.quantile(0.90).reset_index(level=0, drop=True)

    range_features = max_features - min_features
    iqr_features = q75_features - q25_features
    q90_q10_range_features = q90_features - q10_features

    first_features = sorted_df.groupby("unit_id")[extended_agg_feature_cols].shift(extended_agg_window_size - 1)
    if not require_full_window:
        first_available_values = sorted_df.groupby("unit_id")[extended_agg_feature_cols].transform("first")
        first_features = first_features.fillna(first_available_values)

    last_features = sorted_df[extended_agg_feature_cols]
    delta_features = last_features - first_features
    last_minus_mean_features = last_features - mean_features
    relative_delta_features = delta_features / (first_features.abs() + eps)
    relative_last_minus_mean_features = last_minus_mean_features / (mean_features.abs() + eps)

    mean_last_10_features = edge_rolling_features.mean().reset_index(level=0, drop=True)
    std_last_10_features = edge_rolling_features.std(ddof=0).reset_index(level=0, drop=True).fillna(0)
    mean_first_10_features = rolling_features.apply(
        calculate_first_edge_mean,
        raw=True,
    ).reset_index(level=0, drop=True)
    mean_last_10_minus_first_10_features = mean_last_10_features - mean_first_10_features

    slope_features = rolling_features.apply(
        calculate_slope,
        raw=True,
    ).reset_index(level=0, drop=True)
    mad_features = rolling_features.apply(
        calculate_mad,
        raw=True,
    ).reset_index(level=0, drop=True)
    rms_features = rolling_features.apply(
        calculate_rms,
        raw=True,
    ).reset_index(level=0, drop=True)
    slope_change_features = rolling_features.apply(
        calculate_slope_change,
        raw=True,
    ).reset_index(level=0, drop=True)

    feature_blocks = {
        "mean": mean_features,
        "std": std_features,
        "min": min_features,
        "max": max_features,
        "range": range_features,
        "delta": delta_features,
        "slope": slope_features,
        "last_minus_mean": last_minus_mean_features,
        "median": median_features,
        "iqr": iqr_features,
        "last": last_features,
        "first": first_features,
        "relative_delta": relative_delta_features,
        "relative_last_minus_mean": relative_last_minus_mean_features,
        "mean_last_10": mean_last_10_features,
        "mean_first_10": mean_first_10_features,
        "mean_last_10_minus_first_10": mean_last_10_minus_first_10_features,
        "std_last_10": std_last_10_features,
        "q10": q10_features,
        "q90": q90_features,
        "q90_q10_range": q90_q10_range_features,
        "mad": mad_features,
        "rms": rms_features,
        "slope_change": slope_change_features,
    }

    metadata_cols = ["unit_id", "cycle"]
    if include_target:
        metadata_cols.append(target_col)

    renamed_feature_blocks = {
        suffix: rename_feature_block(block, suffix)
        for suffix, block in feature_blocks.items()
    }

    result_df = pd.concat(
        [sorted_df[metadata_cols], *renamed_feature_blocks.values()],
        axis=1,
    )

    if require_full_window:
        result_df = result_df[result_df["cycle"] >= extended_agg_window_size].dropna()

    result_df = result_df.reset_index(drop=True)
    feature_block_columns = {
        suffix: list(block.columns)
        for suffix, block in renamed_feature_blocks.items()
    }

    return result_df, feature_block_columns


In [24]:
print("Building train aggregated features...")
all_agg_window_train_df, feature_block_columns = build_all_agg_window_features(
    train_df,
    include_target=True,
    require_full_window=True,
)
print("Building test aggregated features...")
all_agg_window_test_df, _ = build_all_agg_window_features(
    test_df,
    include_target=False,
    require_full_window=False,
)

all_agg_window_test_last_rows = (
    all_agg_window_test_df
    .sort_values(["unit_id", "cycle"])
    .groupby("unit_id")
    .tail(1)
    .sort_values("unit_id")
    .reset_index(drop=True)
)


def feature_columns_for_blocks(block_names):
    selected_columns = []
    for block_name in block_names:
        selected_columns.extend(feature_block_columns[block_name])
    return selected_columns


best_week2_feature_cols = feature_columns_for_blocks(best_week2_blocks)

X_train_best_week2 = all_agg_window_train_df[best_week2_feature_cols]
y_train_best_week2 = all_agg_window_train_df[target_col].reset_index(drop=True)
X_test_best_week2 = all_agg_window_test_last_rows[best_week2_feature_cols]
y_test_best_week2 = rul_test_df["RUL"].reset_index(drop=True)

if len(X_test_best_week2) != len(y_test_best_week2):
    raise ValueError("test feature rows must match RUL target rows")

print(f"aggregated train shape: {X_train_best_week2.shape}")
print(f"aggregated test shape: {X_test_best_week2.shape}")


Building train aggregated features...
Building test aggregated features...
aggregated train shape: (15731, 432)
aggregated test shape: (100, 432)


## 4. Combined aggregate + lag features

Здесь строим те же комбинированные признаки, что и для CatBoost: агрегаты week2 плюс лаговое окно разного размера. Окно `30` повторяет идею Дениса, окна `10` и `20` проверяют более короткую динамику.

Для лаговой части исключаем те же сенсоры Дениса: `sensor_19`, `sensor_9`, `sensor_3`, `sensor_1`, `sensor_5`, `sensor_10`, `sensor_7`.


In [25]:
denis_excluded_sensors_by_number = [19, 9, 3, 1, 5, 10, 7]
combined_lag_feature_configs = [
    {
        "feature_config": "agg_plus_lag_10_denis_sensors",
        "lag_window_size": 10,
        "excluded_sensors_by_number": denis_excluded_sensors_by_number,
    },
    {
        "feature_config": "agg_plus_lag_20_denis_sensors",
        "lag_window_size": 20,
        "excluded_sensors_by_number": denis_excluded_sensors_by_number,
    },
    {
        "feature_config": "agg_plus_lag_30_denis_sensors",
        "lag_window_size": 30,
        "excluded_sensors_by_number": denis_excluded_sensors_by_number,
    },
]


def selected_lag_feature_cols(excluded_sensors_by_number):
    excluded_sensor_cols = {
        f"sensor_{int(sensor_num)}"
        for sensor_num in excluded_sensors_by_number
    }
    excluded_sensor_cols = sorted(col for col in excluded_sensor_cols if col in feature_cols)
    selected_cols = [col for col in feature_cols if col not in excluded_sensor_cols]
    if not selected_cols:
        raise ValueError("No lag features left after sensor exclusions")
    return selected_cols, excluded_sensor_cols


def build_lag_window_features(source_df, selected_feature_cols, lag_window_size, include_target=False):
    sorted_df = source_df.sort_values(["unit_id", "cycle"]).reset_index(drop=True)
    first_values = sorted_df.groupby("unit_id")[selected_feature_cols].transform("first")
    lagged_parts = []

    for lag in range(lag_window_size):
        shifted = sorted_df.groupby("unit_id")[selected_feature_cols].shift(lag)
        shifted = shifted.fillna(first_values)
        suffix = "t" if lag == 0 else f"t_minus_{lag}"
        shifted.columns = [f"{col}_{suffix}" for col in selected_feature_cols]
        lagged_parts.append(shifted)

    metadata_cols = ["unit_id", "cycle"]
    if include_target:
        metadata_cols.append(target_col)

    return pd.concat([sorted_df[metadata_cols], *lagged_parts], axis=1).reset_index(drop=True)


def build_combined_feature_set(config):
    lag_window_size = config["lag_window_size"]
    lag_base_cols, excluded_sensor_cols = selected_lag_feature_cols(
        config["excluded_sensors_by_number"]
    )

    print(f"Building lag features for {config['feature_config']}...")
    lag_train_df = build_lag_window_features(
        train_df,
        selected_feature_cols=lag_base_cols,
        lag_window_size=lag_window_size,
        include_target=False,
    )
    lag_test_df = build_lag_window_features(
        test_df,
        selected_feature_cols=lag_base_cols,
        lag_window_size=lag_window_size,
        include_target=False,
    )

    lag_feature_cols = [
        col for col in lag_train_df.columns
        if col not in ["unit_id", "cycle", target_col]
    ]

    train_lag_aligned = all_agg_window_train_df[["unit_id", "cycle"]].merge(
        lag_train_df[["unit_id", "cycle", *lag_feature_cols]],
        on=["unit_id", "cycle"],
        how="left",
        validate="one_to_one",
    )
    test_lag_last_rows = (
        lag_test_df
        .sort_values(["unit_id", "cycle"])
        .groupby("unit_id")
        .tail(1)
        .sort_values("unit_id")
        .reset_index(drop=True)
    )

    if train_lag_aligned[lag_feature_cols].isna().any().any():
        raise ValueError(f"Lag train features contain missing values for {config['feature_config']}")
    if len(test_lag_last_rows) != len(y_test_best_week2):
        raise ValueError(f"Lag test rows mismatch for {config['feature_config']}")

    X_train_combined = pd.concat(
        [
            X_train_best_week2.reset_index(drop=True),
            train_lag_aligned[lag_feature_cols].reset_index(drop=True),
        ],
        axis=1,
    )
    X_test_combined = pd.concat(
        [
            X_test_best_week2.reset_index(drop=True),
            test_lag_last_rows[lag_feature_cols].reset_index(drop=True),
        ],
        axis=1,
    )

    print(
        f"{config['feature_config']}: "
        f"agg_features={X_train_best_week2.shape[1]}, "
        f"lag_features={len(lag_feature_cols)}, "
        f"total_features={X_train_combined.shape[1]}, "
        f"excluded_lag_sensors={excluded_sensor_cols}"
    )

    return {
        "feature_config": config["feature_config"],
        "lag_window_size": lag_window_size,
        "excluded_lag_sensors": excluded_sensor_cols,
        "lag_feature_count": len(lag_feature_cols),
        "total_feature_count": X_train_combined.shape[1],
        "X_train": X_train_combined,
        "X_test": X_test_combined,
        "y_train": y_train_best_week2,
        "y_test": y_test_best_week2,
    }


combined_feature_sets = [
    build_combined_feature_set(config)
    for config in combined_lag_feature_configs
]
print(f"Combined feature sets ready: {len(combined_feature_sets)}")


Building lag features for agg_plus_lag_10_denis_sensors...
agg_plus_lag_10_denis_sensors: agg_features=432, lag_features=170, total_features=602, excluded_lag_sensors=['sensor_1', 'sensor_10', 'sensor_19', 'sensor_3', 'sensor_5', 'sensor_7', 'sensor_9']
Building lag features for agg_plus_lag_20_denis_sensors...
agg_plus_lag_20_denis_sensors: agg_features=432, lag_features=340, total_features=772, excluded_lag_sensors=['sensor_1', 'sensor_10', 'sensor_19', 'sensor_3', 'sensor_5', 'sensor_7', 'sensor_9']
Building lag features for agg_plus_lag_30_denis_sensors...
agg_plus_lag_30_denis_sensors: agg_features=432, lag_features=510, total_features=942, excluded_lag_sensors=['sensor_1', 'sensor_10', 'sensor_19', 'sensor_3', 'sensor_5', 'sensor_7', 'sensor_9']
Combined feature sets ready: 3


## 5. Model grids

Перебираем параметры для двух альтернативных моделей.

In [26]:
decision_tree_param_grid = {
    "max_depth": [4, 6, 8, None],
    "min_samples_leaf": [1, 3, 5],
    "min_samples_split": [2, 5, 10],
    "splitter": ["random"],
}


def make_model_specs():
    model_specs = []

    for params in ParameterGrid(decision_tree_param_grid):
        model_specs.append({
            "model_family": "RandomDecisionTreeRegressor",
            "model_name": "DecisionTreeRegressor(splitter=random)",
            "params": params,
            "model": DecisionTreeRegressor(random_state=RANDOM_STATE, **params),
        })

    model_specs.append({
        "model_family": "LinearRegression",
        "model_name": "Scaled LinearRegression",
        "params": {},
        "model": make_pipeline(StandardScaler(), LinearRegression()),
    })

    return model_specs


model_specs = make_model_specs()
total_model_runs = len(combined_feature_sets) * len(model_specs)
if MAX_MODELS_TO_RUN is not None:
    total_model_runs = min(total_model_runs, MAX_MODELS_TO_RUN)

print(f"Feature configs to test: {len(combined_feature_sets)}")
print(f"Random DecisionTree params per feature config: {len(list(ParameterGrid(decision_tree_param_grid)))}")
print("LinearRegression params per feature config: 1")
print(f"Total planned model fits: {len(combined_feature_sets) * len(model_specs)}")
print(f"Total model fits for this run: {total_model_runs}")


Feature configs to test: 3
Random DecisionTree params per feature config: 36
LinearRegression params per feature config: 1
Total planned model fits: 111
Total model fits for this run: 111


## 6. Model search

Эта ячейка обучает все оставшиеся модели на каждом наборе комбинированных признаков, считает official test метрики и сохраняет таблицу результатов.

Кроме MAE, RMSE и R2 считаем ещё direction ошибок: mean_error, долю завышений, max_overestimation и ошибки в near-failure / warning-zone. Это нужно, потому что для RUL важна не только средняя ошибка, но и то, в какую сторону модель ошибается.


In [27]:
def evaluate_predictions(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    error = y_pred - y_true
    near_failure_mask = y_true <= 50
    warning_zone_mask = y_true <= 100
    overestimation = error[error > 0]

    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "r2": r2_score(y_true, y_pred),
        "mean_error": float(error.mean()),
        "median_error": float(np.median(error)),
        "overestimation_share": float((error > 0).mean()),
        "max_overestimation": float(overestimation.max()) if len(overestimation) else 0.0,
        "negative_prediction_count": int((y_pred < 0).sum()),
        "near_failure_mae": mean_absolute_error(y_true[near_failure_mask], y_pred[near_failure_mask]),
        "warning_zone_mae": mean_absolute_error(y_true[warning_zone_mask], y_pred[warning_zone_mask]),
    }


def evaluate_model(feature_set, model_spec, model_number, total_models):
    model_family = model_spec["model_family"]
    model_name = model_spec["model_name"]
    params = model_spec["params"]
    model = model_spec["model"]

    print(
        f"Training {model_number}/{total_models}: "
        f"family={model_family}, feature_config={feature_set['feature_config']}, params={params}"
    )
    start = time.perf_counter()
    model.fit(feature_set["X_train"], feature_set["y_train"])
    train_seconds = time.perf_counter() - start

    y_pred = model.predict(feature_set["X_test"])
    metrics = evaluate_predictions(feature_set["y_test"], y_pred)

    result = {
        "model_family": model_family,
        "model_name": model_name,
        "feature_config": feature_set["feature_config"],
        "lag_window_size": feature_set["lag_window_size"],
        "lag_feature_count": feature_set["lag_feature_count"],
        "total_feature_count": feature_set["total_feature_count"],
        "train_seconds": train_seconds,
        **params,
        **metrics,
    }
    print(
        f"Finished {model_number}/{total_models}: "
        f"MAE={result['mae']:.6f}, RMSE={result['rmse']:.6f}, R2={result['r2']:.6f}, "
        f"over_share={result['overestimation_share']:.3f}, seconds={train_seconds:.1f}"
    )
    return result


experiment_results = []
model_number = 0
stop_requested = False

for feature_set in combined_feature_sets:
    for model_spec in make_model_specs():
        if MAX_MODELS_TO_RUN is not None and model_number >= MAX_MODELS_TO_RUN:
            stop_requested = True
            break
        model_number += 1
        experiment_results.append(
            evaluate_model(
                feature_set=feature_set,
                model_spec=model_spec,
                model_number=model_number,
                total_models=total_model_runs,
            )
        )
    if stop_requested:
        break

experiment_results_df = (
    pd.DataFrame(experiment_results)
    .sort_values(["mae", "rmse"])
    .reset_index(drop=True)
)

output_path = OUTPUT_DIR / "lev_experiments_results.csv"
experiment_results_df.to_csv(output_path, index=False)
print(f"Saved results to {output_path}")
experiment_results_df.head(20)


Training 1/111: family=RandomDecisionTreeRegressor, feature_config=agg_plus_lag_10_denis_sensors, params={'max_depth': 4, 'min_samples_leaf': 1, 'min_samples_split': 2, 'splitter': 'random'}
Finished 1/111: MAE=17.823881, RMSE=24.078742, R2=0.664256, over_share=0.620, seconds=0.1
Training 2/111: family=RandomDecisionTreeRegressor, feature_config=agg_plus_lag_10_denis_sensors, params={'max_depth': 4, 'min_samples_leaf': 1, 'min_samples_split': 5, 'splitter': 'random'}
Finished 2/111: MAE=17.823881, RMSE=24.078742, R2=0.664256, over_share=0.620, seconds=0.1
Training 3/111: family=RandomDecisionTreeRegressor, feature_config=agg_plus_lag_10_denis_sensors, params={'max_depth': 4, 'min_samples_leaf': 1, 'min_samples_split': 10, 'splitter': 'random'}
Finished 3/111: MAE=17.823881, RMSE=24.078742, R2=0.664256, over_share=0.620, seconds=0.1
Training 4/111: family=RandomDecisionTreeRegressor, feature_config=agg_plus_lag_10_denis_sensors, params={'max_depth': 4, 'min_samples_leaf': 3, 'min_sample

,model_family,model_name,feature_config,lag_window_size,lag_feature_count,total_feature_count,train_seconds,max_depth,min_samples_leaf,min_samples_split,...,mae,rmse,r2,mean_error,median_error,overestimation_share,max_overestimation,negative_prediction_count,near_failure_mae,warning_zone_mae
0,RandomDecisionTreeRegressor,DecisionTreeRegressor(splitter=random),agg_plus_lag_10_denis_sensors,10,170,602,0.169812,8.0,1.0,5.0,...,15.832511,23.351176,0.684239,6.061909,2.931868,0.61,101.772727,0,5.006126,12.564189
1,RandomDecisionTreeRegressor,DecisionTreeRegressor(splitter=random),agg_plus_lag_10_denis_sensors,10,170,602,0.169722,8.0,3.0,10.0,...,16.624777,26.225081,0.601733,8.299540,4.730000,0.64,150.000000,0,5.215467,16.536841
2,RandomDecisionTreeRegressor,DecisionTreeRegressor(splitter=random),agg_plus_lag_10_denis_sensors,10,170,602,0.130179,6.0,1.0,2.0,...,16.803724,22.206855,0.714429,5.656592,5.523783,0.61,78.026936,0,7.308556,14.984966
3,RandomDecisionTreeRegressor,DecisionTreeRegressor(splitter=random),agg_plus_lag_10_denis_sensors,10,170,602,0.179951,8.0,3.0,2.0,...,16.853478,25.135689,0.634134,9.009279,2.324163,0.59,91.132565,0,5.691176,14.829650
4,RandomDecisionTreeRegressor,DecisionTreeRegressor(splitter=random),agg_plus_lag_10_denis_sensors,10,170,602,0.173597,8.0,3.0,5.0,...,16.853478,25.135689,0.634134,9.009279,2.324163,0.59,91.132565,0,5.691176,14.829650
5,RandomDecisionTreeRegressor,DecisionTreeRegressor(splitter=random),agg_plus_lag_10_denis_sensors,10,170,602,0.181840,8.0,1.0,10.0,...,16.977341,26.261544,0.600625,7.613936,4.357100,0.62,150.000000,0,7.007850,17.586587
6,RandomDecisionTreeRegressor,DecisionTreeRegressor(splitter=random),agg_plus_lag_10_denis_sensors,10,170,602,0.140254,6.0,1.0,5.0,...,17.402305,23.839549,0.670893,7.574633,5.081263,0.65,66.858257,0,8.104729,13.882934
7,RandomDecisionTreeRegressor,DecisionTreeRegressor(splitter=random),agg_plus_lag_10_denis_sensors,10,170,602,0.125401,6.0,1.0,10.0,...,17.402305,23.839549,0.670893,7.574633,5.081263,0.65,66.858257,0,8.104729,13.882934
8,RandomDecisionTreeRegressor,DecisionTreeRegressor(splitter=random),agg_plus_lag_10_denis_sensors,10,170,602,0.127159,6.0,3.0,2.0,...,17.402305,23.839549,0.670893,7.574633,5.081263,0.65,66.858257,0,8.104729,13.882934
9,RandomDecisionTreeRegressor,DecisionTreeRegressor(splitter=random),agg_plus_lag_10_denis_sensors,10,170,602,0.127724,6.0,3.0,5.0,...,17.402305,23.839549,0.670893,7.574633,5.081263,0.65,66.858257,0,8.104729,13.882934


## 7. Best results by model family

Смотрим лучший вариант внутри каждого семейства моделей. Эта таблица нужна для короткого вывода в проекте: что пробовали и какие ошибки получили.


In [28]:
required_columns = [
    "model_family",
    "feature_config",
    "lag_window_size",
    "total_feature_count",
    "train_seconds",
    "mae",
    "rmse",
    "r2",
    "mean_error",
    "overestimation_share",
    "max_overestimation",
    "negative_prediction_count",
    "near_failure_mae",
    "warning_zone_mae",
    "n_estimators",
    "learning_rate",
    "max_depth",
    "min_samples_leaf",
    "min_samples_split",
]

available_columns = [col for col in required_columns if col in experiment_results_df.columns]
best_by_family_df = (
    experiment_results_df
    .sort_values(["model_family", "mae", "rmse"])
    .groupby("model_family", as_index=False)
    .head(1)
    .sort_values("mae")
    .reset_index(drop=True)
)
best_by_family_df[available_columns]


,model_family,feature_config,lag_window_size,total_feature_count,train_seconds,mae,rmse,r2,mean_error,overestimation_share,max_overestimation,negative_prediction_count,near_failure_mae,warning_zone_mae,max_depth,min_samples_leaf,min_samples_split
0,RandomDecisionTreeRegressor,agg_plus_lag_10_denis_sensors,10,602,0.169812,15.832511,23.351176,0.684239,6.061909,0.61,101.772727,0,5.006126,12.564189,8.0,1.0,5.0
1,LinearRegression,agg_plus_lag_10_denis_sensors,10,602,0.366807,19.156566,24.199950,0.660868,7.630267,0.64,94.073775,6,15.116179,18.181331,NaN,NaN,NaN


## 8. Error tables for the current best models

Короткая таблица ошибок для текста проекта. После запуска здесь будет видно, насколько одиночное дерево и линейная регрессия проиграли или выиграли относительно ожиданий.


In [29]:
display_columns = [
    "model_family",
    "feature_config",
    "lag_window_size",
    "mae",
    "rmse",
    "r2",
    "mean_error",
    "overestimation_share",
    "max_overestimation",
    "near_failure_mae",
    "warning_zone_mae",
    "train_seconds",
]

best_by_family_df[display_columns].style.format({
    "mae": "{:.3f}",
    "rmse": "{:.3f}",
    "r2": "{:.3f}",
    "mean_error": "{:.3f}",
    "overestimation_share": "{:.3f}",
    "max_overestimation": "{:.3f}",
    "near_failure_mae": "{:.3f}",
    "warning_zone_mae": "{:.3f}",
    "train_seconds": "{:.1f}",
})


,model_family,feature_config,lag_window_size,mae,rmse,r2,mean_error,overestimation_share,max_overestimation,near_failure_mae,warning_zone_mae,train_seconds
0,RandomDecisionTreeRegressor,agg_plus_lag_10_denis_sensors,10,15.833,23.351,0.684,6.062,0.610,101.773,5.006,12.564,0.2
1,LinearRegression,agg_plus_lag_10_denis_sensors,10,19.157,24.200,0.661,7.630,0.640,94.074,15.116,18.181,0.4


## 9. Generated conclusion

Эта ячейка после запуска сама собирает короткий вывод по фактическим ошибкам. Его можно использовать в описании проекта, но числа лучше проверить глазами.


In [30]:
def result_sentence(row):
    return (
        f"- **{row['model_family']}**: best MAE={row['mae']:.3f}, "
        f"RMSE={row['rmse']:.3f}, R2={row['r2']:.3f}; "
        f"feature_config=`{row['feature_config']}`, lag_window={int(row['lag_window_size'])}, "
        f"overestimation_share={row['overestimation_share']:.3f}, "
        f"near_failure_MAE={row['near_failure_mae']:.3f}."
    )


def generated_conclusion(results_df):
    if results_df.empty:
        return "No models were evaluated yet. Run the search cell first."

    best_overall = results_df.sort_values(["mae", "rmse"]).iloc[0]
    best_by_family = (
        results_df
        .sort_values(["model_family", "mae", "rmse"])
        .groupby("model_family", as_index=False)
        .head(1)
        .sort_values("mae")
    )

    lines = [
        "### Итоговый вывод",
        "",
        "В этом ноутбуке проверены простые альтернативы не-RandomForest на той же стадии, где пробовали CatBoost: combined Week 2 aggregated features плюс raw lag windows, без conservative shift.",
        "",
        f"Лучший общий результат здесь дала модель **{best_overall['model_family']}**: MAE={best_overall['mae']:.3f}, RMSE={best_overall['rmse']:.3f}, R2={best_overall['r2']:.3f}.",
        "",
        "Лучшие результаты по семействам:",
    ]
    lines.extend(result_sentence(row) for _, row in best_by_family.iterrows())
    lines.extend([
        "",
        "Интерпретация: если эти MAE остаются хуже Week 2 / Week 3 RandomForest baseline, эксперимент подтверждает формулировку проекта: разные семейства моделей были проверены, но RandomForest остался самым сильным практическим вариантом. Одиночное случайное дерево ожидаемо менее устойчиво, а LinearRegression может недообучать нелинейную деградацию. Gradient Boosting здесь не запускался из-за времени обучения; бустинг отдельно проверялся через CatBoost.",
        "",
        "Shift-калибровки здесь нет, поэтому сравнение остаётся на уровне CatBoost-era direct model search, а не более поздних Week 4 safety-calibrated экспериментов.",
    ])
    return "\n".join(lines)


display(Markdown(generated_conclusion(experiment_results_df)))


### Итоговый вывод

В этом ноутбуке проверены простые альтернативы не-RandomForest на той же стадии, где пробовали CatBoost: combined Week 2 aggregated features плюс raw lag windows, без conservative shift.

Лучший общий результат здесь дала модель **RandomDecisionTreeRegressor**: MAE=15.833, RMSE=23.351, R2=0.684.

Лучшие результаты по семействам:
- **RandomDecisionTreeRegressor**: best MAE=15.833, RMSE=23.351, R2=0.684; feature_config=`agg_plus_lag_10_denis_sensors`, lag_window=10, overestimation_share=0.610, near_failure_MAE=5.006.
- **LinearRegression**: best MAE=19.157, RMSE=24.200, R2=0.661; feature_config=`agg_plus_lag_10_denis_sensors`, lag_window=10, overestimation_share=0.640, near_failure_MAE=15.116.

Интерпретация: если эти MAE остаются хуже Week 2 / Week 3 RandomForest baseline, эксперимент подтверждает формулировку проекта: разные семейства моделей были проверены, но RandomForest остался самым сильным практическим вариантом. Одиночное случайное дерево ожидаемо менее устойчиво, а LinearRegression может недообучать нелинейную деградацию. Gradient Boosting здесь не запускался из-за времени обучения; бустинг отдельно проверялся через CatBoost.

Shift-калибровки здесь нет, поэтому сравнение остаётся на уровне CatBoost-era direct model search, а не более поздних Week 4 safety-calibrated экспериментов.